In [98]:
import pandas as pd

# Read the HTML file
tables = pd.read_html('holidays.html')

# Assuming the first table in the file is the one you want
holidays_23, holidays_22 = tables

# Write the DataFrame to a CSV file
holidays_23.to_csv('holidays_23.csv', index=False)
holidays_22.to_csv('holidays_22.csv', index=False)

In [99]:
holidays_22 = pd.read_csv('holidays_22.csv', index_col=False)
holidays_23 = pd.read_csv('holidays_23.csv', index_col=False)

In [100]:
holidays_22.dropna(inplace=True)
holidays_23.dropna(inplace=True)

In [101]:
# print shape of holidays_22
print(holidays_22.shape)
print(holidays_23.shape)

(78, 4)
(79, 4)


In [102]:
holidays_22.info()

<class 'pandas.core.frame.DataFrame'>
Index: 78 entries, 0 to 89
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Date                78 non-null     object
 1   Unnamed: 1_level_0  78 non-null     object
 2   Name                78 non-null     object
 3   Type                78 non-null     object
dtypes: object(4)
memory usage: 3.0+ KB


In [103]:
!pip install regex


[notice] A new release of pip is available: 23.0.1 -> 23.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [125]:
# for the 2022 holidays, add datetime index - Date column is a string of format date and month ex: 01 January convert it to datetime index
import regex as re
import datetime as datetime

date = []
month_mapping = {
    'Jan' : '01',
    'Feb' : '02',
    'Mar' : '03',
    'Apr' : '04',
    'May' : '05',
    'Jun' : '06',
    'Jul' : '07',
    'Aug' : '08',
    'Sep' : '09',
    'Oct' : '10',
    'Nov' : '11',
    'Dec' : '12'
    
}

# convert the below into a function
def convert_to_datetime(string, year):
    # string format is 12 Dec - 2 digits followed by 3 letters
    # extracting the date and month
    date = re.findall(r'\d+', string)
    month = re.findall(r'[a-zA-Z]+', string)
    # mapping the month to the month number
    month = month_mapping[month[0]]
    # converting the list to a string
    date = int(date[0])
    month = int(month)
    # creating a datetime object
    date = datetime.datetime(year, month, date)
    # appending the datetime object to a list
    return date

def date_prep(df, year):
    # df drop row with index 0
    df = df.drop(0)
    # df['Date'] = df['Date'].apply(convert_to_datetime( year=year))
    df['Date'] = df['Date'].apply(lambda x: convert_to_datetime(x, year=year))
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d %H:%M:%S.%f', errors='coerce')
    df = df.set_index('Date')

    df['Day'] = df.index.day_name()
    df = df.drop(columns=['Unnamed: 1_level_0'])
    return df
def final_df(df_list, year_list):
    df_final = pd.DataFrame()
    for df, year in zip(df_list, year_list):
        df = date_prep(df, year)
        # concat the df to df_final
        df_final = pd.concat([df_final, df])
        df_final.drop_duplicates(inplace=True)
        # drop same index rows
        df_final = df_final[~df_final.index.duplicated(keep='first')]
    return df_final


In [126]:
df_list = [holidays_22, holidays_23]
year_list = [2022, 2023]
df_final = final_df(df_list, year_list)

In [127]:
df_final.head(10)

,Name,Type,Day
Date,,,
2022-01-01,New Year's Day,Restricted Holiday,Saturday
2022-01-09,Guru Govind Singh Jayanti,Restricted Holiday,Sunday
2022-01-13,Lohri,Restricted Holiday,Thursday
2022-01-14,Pongal,Restricted Holiday,Friday
2022-01-26,Republic Day,Gazetted Holiday,Wednesday
2022-02-01,Lunar New Year,Observance,Tuesday
2022-02-05,Vasant Panchami,Restricted Holiday,Saturday
2022-02-14,Valentine's Day,Observance,Monday
2022-02-15,Hazarat Ali's Birthday,Restricted Holiday,Tuesday


In [122]:
df_final.to_csv('holidays.csv')